In [ ]:
"""
Trustpilot Scraper

This script performs the following tasks:
1. Loads initial categories from Trustpilot if the cache (SQLite DB) is empty.
2. Runs sequential scraping passes over unvisited category pages to discover new subcategories.
   If a category page yields no intra‑website links (likely due to rate limiting), an exponential backoff
   mechanism retries the page before eventually marking it as visited.
3. Presents an alphabetized list of cleaned category names for user selection.
4. Prompts the user for filtering options.
5. Scrapes the selected category across all countries (using cached country data) and extracts up to
   three businesses per country matching the filtering criteria (verified, claimed, minimum TrustScore).
6. Business results are cached for 168 hours (7 days) to avoid unnecessary re‑scraping.
7. HTTP responses are additionally cached using requests_cache to minimize repeated requests.
"""

# -----------------------------------------------------------------------------
# Install required packages (if not already installed)
# -----------------------------------------------------------------------------
%pip install -r requirements.txt

# -----------------------------------------------------------------------------
# Imports and Global Modules
# -----------------------------------------------------------------------------
import os
import time
import sqlite3
import requests
import logging
import re
import hashlib
from bs4 import BeautifulSoup
import multiprocessing

# -----------------------------------------------------------------------------
# Global Configuration
# -----------------------------------------------------------------------------
MAX_CATEGORY_ATTEMPTS = 5             # Maximum attempts to retry a category page when no links are found.
INITIAL_BACKOFF_DELAY = 1             # Initial delay for exponential backoff (in seconds)
REQUEST_TIMEOUT = 10                  # Timeout for HTTP requests (in seconds)
STEP_ONE = False                      # Do not run category scraping; use manual categories.
STEP_TWO = True                       # Use manual categories for selection.
MANUAL_CATEGORIES = ["online_pharmacy"]  # List of manual category names.
STEP_THREE = True
STEP_FOUR = True
STEP_FIVE = True
STEP_SIX = True

# -----------------------------------------------------------------------------
# Setup Working Directories, Cache Paths, and Logging
# -----------------------------------------------------------------------------
SCRIPT_DIR = os.getcwd()                                   # Current working directory.
CACHE_DIR = os.path.join(SCRIPT_DIR, "cache")              # Directory for cache files and DB.
DB_PATH = os.path.join(CACHE_DIR, "trustpilot.db")         # SQLite DB file path.
os.makedirs(CACHE_DIR, exist_ok=True)                      # Ensure cache directory exists.

# Configure logging (both file and console).
LOG_FILE = os.path.join(CACHE_DIR, "scraper.log")
logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format="%(asctime)s - %(message)s")

# -----------------------------------------------------------------------------
# OPTIONAL HTTP CACHING via requests_cache
# -----------------------------------------------------------------------------
try:
    import requests_cache
    # Cache GET requests for 1 hour (3600 seconds).
    requests_cache.install_cache(cache_name=os.path.join(CACHE_DIR, 'trustpilot_http_cache'),
                                 backend='sqlite', expire_after=3600)
    CACHE_ACTIVE = True
except ImportError:
    CACHE_ACTIVE = False

# -----------------------------------------------------------------------------
# HTTP Session Setup
# -----------------------------------------------------------------------------
# Create a persistent session with a timeout to reuse TCP connections.
session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/120.0.0.0 Safari/537.36")
})

# -----------------------------------------------------------------------------
# Other Global Constants
# -----------------------------------------------------------------------------
BUSINESS_CACHE_EXPIRY = 168 * 3600  # Business results are fresh for 168 hours (7 days).
CPU_CORES = max(1, multiprocessing.cpu_count() - 1)  # Use one less than available CPU cores.

# -----------------------------------------------------------------------------
# Database Initialization
# -----------------------------------------------------------------------------
def init_db():
    """
    Initialize the SQLite database and create necessary tables:
      - categories: stores category names, URLs, visited status, level, and number of attempts.
      - businesses: caches business results per category and country.
      - countries: caches the list of countries indefinitely.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    # Categories table.
    cur.execute("""
        CREATE TABLE IF NOT EXISTS categories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE,
            url TEXT UNIQUE,
            visited INTEGER DEFAULT 0,
            level INTEGER,
            attempts INTEGER DEFAULT 0
        )
    """)
    # Businesses table.
    cur.execute("""
        CREATE TABLE IF NOT EXISTS businesses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT,
            country TEXT,
            business_name TEXT,
            url TEXT,
            rating REAL,
            claimed INTEGER,
            verified INTEGER,
            params_hash TEXT,
            timestamp INTEGER
        )
    """)
    # Countries table.
    cur.execute("""
        CREATE TABLE IF NOT EXISTS countries (
            code TEXT PRIMARY KEY,
            name TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

# -----------------------------------------------------------------------------
# Logging Helper Function
# -----------------------------------------------------------------------------
def log_message(message):
    """Log a message to both the console and the log file."""
    print(message)
    logging.info(message)

# -----------------------------------------------------------------------------
# Category Name Cleaning and Validity Functions
# -----------------------------------------------------------------------------
def clean_category_name(name):
    """
    Remove any trailing digits (and preceding whitespace) from the category name.
    E.g., "Gaming 240" becomes "Gaming".
    """
    cleaned = re.sub(r'\s*\d+\s*$', '', name)
    return cleaned.strip()

def is_valid_category(name):
    """
    Validate a category name to ensure it is not empty and not solely digits.
    """
    if not name:
        return False
    if name.isdigit():
        return False
    return True

# -----------------------------------------------------------------------------
# Database Helper Functions for Categories
# -----------------------------------------------------------------------------
def get_all_categories():
    """Return a set of all known category names from the database."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name FROM categories")
    cats = {row[0] for row in cur.fetchall()}
    conn.close()
    return cats

def get_all_categories_flat():
    """
    Return a list of (name, url) tuples for all valid (cleaned) categories,
    sorted alphabetically.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name, url FROM categories")
    cats = cur.fetchall()
    conn.close()
    valid_cats = [(name, url) for name, url in cats if is_valid_category(name)]
    return sorted(valid_cats, key=lambda x: x[0].lower())

def get_unvisited_categories(level):
    """
    Retrieve all categories at a given level that have not been visited.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name, url FROM categories WHERE visited = 0 AND level = ?", (level,))
    categories = cur.fetchall()
    conn.close()
    return categories

def mark_category_visited(name):
    """Mark a given category as visited in the database."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("UPDATE categories SET visited = 1 WHERE name = ?", (name,))
    conn.commit()
    conn.close()

def save_category(name, url, level, known_categories):
    """
    Clean and validate the category name. If valid and not already known, save it to the database.
    """
    cleaned_name = clean_category_name(name)
    if not is_valid_category(cleaned_name):
        return
    if cleaned_name in known_categories:
        return
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("INSERT OR IGNORE INTO categories (name, url, visited, level) VALUES (?, ?, 0, ?)",
                (cleaned_name, url, level))
    conn.commit()
    conn.close()
    known_categories.add(cleaned_name)

def get_visited_counts():
    """
    Return a tuple (visited_count, total_count) of categories in the database.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM categories WHERE visited = 1")
    visited = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM categories")
    total = cur.fetchone()[0]
    conn.close()
    return visited, total

def increment_attempts(name):
    """
    Increment the attempt count for a given category and return the updated count.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("UPDATE categories SET attempts = attempts + 1 WHERE name = ?", (name,))
    conn.commit()
    cur.execute("SELECT attempts FROM categories WHERE name = ?", (name,))
    attempts = cur.fetchone()[0]
    conn.close()
    return attempts

# -----------------------------------------------------------------------------
# (Not used in manual mode) Category Scraping Functions
# -----------------------------------------------------------------------------
def initialize_categories():
    """
    If no categories exist in the database, fetch initial level 1 categories from Trustpilot.
    """
    if get_all_categories_flat():
        return  # Already initialized
    log_message("🔄 No categories found in cache. Fetching initial Level 1 categories from Trustpilot...")
    try:
        response = session.get("https://www.trustpilot.com/categories", timeout=REQUEST_TIMEOUT)
    except Exception as e:
        log_message(f"⚠ Error fetching initial categories: {e}")
        return
    if response.status_code != 200:
        log_message(f"⚠ Failed to fetch initial categories, status code: {response.status_code}")
        return
    soup = BeautifulSoup(response.text, "html.parser")
    category_links = soup.select("a[href^='/categories/']")
    known_categories = get_all_categories()
    for link in category_links:
        raw_name = link.text.strip()
        url = f"https://www.trustpilot.com{link['href']}"
        save_category(raw_name, url, level=1, known_categories=known_categories)
    log_message("✅ Initial Level 1 categories discovered and saved.")

def scrape_category(name, url, level, known_categories):
    """
    Open the category URL and scrape for subcategory links.
    Retry using exponential backoff if no subcategories are found.
    """
    attempts = 0
    delay = INITIAL_BACKOFF_DELAY
    found_categories = set()
    visited_categories = set()

    while attempts < MAX_CATEGORY_ATTEMPTS:
        log_message(f"\n📂 Opening category: {name} (Attempt {attempts+1})")
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
        except Exception as e:
            log_message(f"⚠ Error fetching {url}: {e}")
            attempts += 1
            time.sleep(delay)
            delay *= 2
            continue
        if response.status_code != 200:
            log_message(f"⚠ Failed to fetch {name} (status code {response.status_code})")
            attempts += 1
            time.sleep(delay)
            delay *= 2
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        sub_links = soup.select("a[href^='/categories/']")
        found_categories = set()
        visited_categories = set()

        for sub in sub_links:
            raw_sub_name = sub.text.strip()
            cleaned_sub_name = clean_category_name(raw_sub_name)
            if not is_valid_category(cleaned_sub_name):
                continue
            sub_url = f"https://www.trustpilot.com{sub['href']}"
            if cleaned_sub_name not in known_categories:
                save_category(raw_sub_name, sub_url, level + 1, known_categories)
                found_categories.add(cleaned_sub_name)
            else:
                visited_categories.add(cleaned_sub_name)

        if found_categories or visited_categories:
            log_message(f"✅ Category {name} yielded {len(found_categories)} new and {len(visited_categories)} known subcategories.")
            break
        else:
            attempts += 1
            log_message(f"⚠ No subcategories found for {name}. Likely rate limited. Retrying in {delay} seconds...")
            time.sleep(delay)
            delay *= 2

    mark_category_visited(name)
    return name, found_categories, visited_categories

def run_category_scraping_run():
    """
    Process unvisited categories at levels 1 and 2 sequentially.
    """
    known_categories = get_all_categories()
    for level in range(1, 3):
        unvisited = get_unvisited_categories(level)
        if not unvisited:
            log_message(f"✅ No unvisited Level {level} categories in this run.")
            continue
        log_message(f"\n🔄 Fetching Level {level + 1} subcategories sequentially...")
        for name, url in unvisited:
            cat_name, found, visited = scrape_category(name, url, level, known_categories)
            v_count, t_count = get_visited_counts()
            progress = (v_count / t_count * 100) if t_count > 0 else 0
            log_message(f"✅ {cat_name} - Found {len(found)} new, {len(visited)} visited subcategories.")
            log_message(f"   Overall Progress: {v_count}/{t_count} categories visited ({progress:.1f}%)")
            for subcat in found:
                log_message(f"   ➡ {subcat}")
            time.sleep(5)

def scrape_categories_main():
    """
    Perform three sequential scraping runs for categories.
    (Not used when using manual categories.)
    """
    initialize_categories()  # Ensure initial Level 1 categories exist.
    for run in range(3):
        log_message(f"\n🚀 Starting scraping run {run + 1} for categories...")
        run_category_scraping_run()
        time.sleep(5)  # Short pause between runs.

# -----------------------------------------------------------------------------
# Modified Country Caching Function
# -----------------------------------------------------------------------------
def get_countries(source_url=None):
    """
    Retrieve the list of countries (code, name) from the database.
    If not cached, fetch the page source from the specified URL and use BeautifulSoup
    to find the <select> element with data-country-selector-filter-select="true".
    
    Parameters:
      - source_url (str): URL to fetch the country selector from. If None, defaults to the homepage.
    """
    # Connect to the database and try to fetch cached countries.
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT code, name FROM countries")
    countries = cur.fetchall()
    if countries:
        conn.close()
        return countries

    # Use the source_url if provided; otherwise default to the homepage.
    if source_url is None:
        url = "https://www.trustpilot.com/"
    else:
        url = source_url

    log_message(f"🔄 Fetching list of countries from {url} ...")
    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
    except Exception as e:
        log_message(f"⚠ Error fetching page {url}: {e}")
        conn.close()
        return []
    
    if response.status_code != 200:
        log_message(f"⚠ Failed to fetch page {url} (status code: {response.status_code})")
        conn.close()
        return []
    
    # Parse the HTML using BeautifulSoup.
    soup = BeautifulSoup(response.text, "html.parser")
    # Use a CSS selector to locate the <select> element with the attribute data-country-selector-filter-select="true".
    select = soup.select_one("select[data-country-selector-filter-select='true']")
    if not select:
        log_message("⚠ Could not find country selector on the page.")
        conn.close()
        return []
    
    # Extract all <option> elements from the <select>.
    options = select.find_all("option")
    countries = []
    for option in options:
        code = option.get("value", "").strip()
        name = option.text.strip()
        if code and name:
            countries.append((code, name))
            cur.execute("INSERT OR IGNORE INTO countries (code, name) VALUES (?, ?)", (code, name))
    
    conn.commit()
    conn.close()
    log_message(f"✅ Cached {len(countries)} countries.")
    return countries

# -----------------------------------------------------------------------------
# Business Caching Helper Functions
# -----------------------------------------------------------------------------
def is_business_cache_valid(category, param_hash, country):
    """
    Check whether business results for the given (category, country, parameters)
    were scraped within the cache expiry period.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        SELECT MAX(timestamp) FROM businesses
        WHERE category = ? AND params_hash = ? AND country = ?
    """, (category, param_hash, country))
    last_scraped = cur.fetchone()[0]
    conn.close()
    if last_scraped is None:
        return False
    return (time.time() - last_scraped) < BUSINESS_CACHE_EXPIRY

def save_businesses(category, country, businesses, param_hash):
    """
    Save a list of business records for the given category and country in the database.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    for biz in businesses:
        cur.execute("""
            INSERT INTO businesses (category, country, business_name, url, rating, claimed, verified, params_hash, timestamp)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (category, country, biz["name"], biz["url"], biz["rating"],
              biz["claimed"], biz["verified"], param_hash, int(time.time())))
    conn.commit()
    conn.close()

# -----------------------------------------------------------------------------
# New Helper Function to Mark a Failed Query
# -----------------------------------------------------------------------------
def mark_query_failed(category, country, param_hash):
    """
    Insert a dummy record into the businesses table to mark that a query with the given
    parameter set (category, country, and parameter hash) has failed (HTTP 404).
    This record will be used by the caching mechanism so that the same query is not attempted again
    for 168 hours.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    # Insert a dummy record with a unique business name that indicates a failed query.
    cur.execute("""
        INSERT INTO businesses (category, country, business_name, url, rating, claimed, verified, params_hash, timestamp)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (category, country, "__FAILED__", "", 0.0, 0, 0, param_hash, int(time.time())))
    conn.commit()
    conn.close()

# -----------------------------------------------------------------------------
# Business Scraping per Country
# -----------------------------------------------------------------------------
def scrape_top_businesses_for_country(category_name, base_category_url, country, verified, claimed, min_trustscore):
    """
    Build the URL for a given category and country by appending query parameters.
    Always include &sort=reviews_count to sort results by review count.
    Optionally include a trustscore parameter if a minimum TrustScore is provided.
    Scrape the page and extract up to 3 businesses matching the filtering criteria.
    
    If the HTTP response returns a 404 status code, mark that query as failed (visited)
    so that it is not re-queried for 168 hours.
    
    Parameters:
      - category_name (str): The name of the category.
      - base_category_url (str): The base URL for the category.
      - country (str): The country code.
      - verified (bool): Filter for verified businesses (not used in extraction, default False).
      - claimed (bool): Filter for claimed businesses (not used in extraction, default False).
      - min_trustscore (float or None): The minimum TrustScore required.
    
    Returns:
      - A list of up to 3 dictionaries containing business details.
    """
    # -----------------------------------------------------------------------------
    # Build the target URL:
    # Start with the base URL, append the country parameter and the sort parameter.
    # -----------------------------------------------------------------------------
    url = f"{base_category_url}?country={country}&sort=reviews_count"
    
    # Append the trustscore parameter if provided.
    if min_trustscore is not None:
        url += f"&trustscore={min_trustscore}"
    
    # -----------------------------------------------------------------------------
    # Create a unique parameter string and its MD5 hash for caching purposes.
    # -----------------------------------------------------------------------------
    param_str = f"{category_name}-{country}-{verified}-{claimed}-{min_trustscore if min_trustscore is not None else 'ANY'}"
    param_hash = hashlib.md5(param_str.encode()).hexdigest()
    
    # -----------------------------------------------------------------------------
    # If results for this query are cached and still valid, return an empty list.
    # -----------------------------------------------------------------------------
    if is_business_cache_valid(category_name, param_hash, country):
        log_message(f"🔄 Using cached results for {category_name} in {country} (verified={verified}, claimed={claimed}, min_trustscore={min_trustscore})")
        return []
    
    log_message(f"\n🔍 Scraping top businesses for: {category_name} in {country} (verified={verified}, claimed={claimed}, min_trustscore={min_trustscore})")
    
    # -----------------------------------------------------------------------------
    # Fetch the page.
    # -----------------------------------------------------------------------------
    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
    except Exception as e:
        log_message(f"⚠ Error fetching {url}: {e}")
        return []
    
    # -----------------------------------------------------------------------------
    # Check if the response is a 404:
    # If so, mark this query as failed so it won't be re-scraped for 168 hours.
    # -----------------------------------------------------------------------------
    if response.status_code == 404:
        mark_query_failed(category_name, country, param_hash)
        log_message(f"⚠ Received 404 for {category_name} in {country} (verified={verified}, claimed={claimed}, min_trustscore={min_trustscore}). Marking as visited for 168 hours.")
        return []
    elif response.status_code != 200:
        log_message(f"⚠ Failed to fetch businesses for {category_name} in {country} (status code {response.status_code})")
        return []
    
    # -----------------------------------------------------------------------------
    # Parse the page using BeautifulSoup.
    # -----------------------------------------------------------------------------
    soup = BeautifulSoup(response.text, "html.parser")
    
    # -----------------------------------------------------------------------------
    # Extract the business URL elements.
    # -----------------------------------------------------------------------------
    url_tags = soup.find_all("p", class_=lambda c: c and "styles_websiteUrlDisplayed__lSw1A" in c)
    
    # -----------------------------------------------------------------------------
    # Extract the rating elements.
    # -----------------------------------------------------------------------------
    rating_tags = soup.find_all("p", class_=lambda c: c and "styles_ratingText__A2dmB" in c)
    
    businesses_raw = []
    # -----------------------------------------------------------------------------
    # Pair each URL tag with its corresponding rating tag.
    # -----------------------------------------------------------------------------
    for url_tag, rating_tag in zip(url_tags, rating_tags):
        # -----------------------------------------------------------------------------
        # Extract the business URL (and use it as the business name).
        # -----------------------------------------------------------------------------
        biz_text = url_tag.get_text(strip=True)
        biz_url = f"https://{biz_text}" if not biz_text.startswith("http") else biz_text
        biz_name = biz_text
        
        # -----------------------------------------------------------------------------
        # Extract the TrustScore and review count from the rating text.
        # -----------------------------------------------------------------------------
        rating_text = rating_tag.get_text(" ", strip=True)
        rating_match = re.search(r"TrustScore\s*([\d\.]+)", rating_text)
        rating = float(rating_match.group(1)) if rating_match else 0.0
        
        review_match = re.search(r"\|\s*([\d,]+)\s+reviews", rating_text)
        if review_match:
            review_count = int(review_match.group(1).replace(",", ""))
        else:
            review_count = 0
        
        # -----------------------------------------------------------------------------
        # Filter by minimum TrustScore if specified.
        # -----------------------------------------------------------------------------
        if min_trustscore is None or rating >= min_trustscore:
            businesses_raw.append({
                "name": biz_name,
                "url": biz_url,
                "rating": rating,
                "review_count": review_count,
                "claimed": False,
                "verified": False,
                "country": country
            })
    
    # -----------------------------------------------------------------------------
    # Sort the businesses by highest review count.
    # -----------------------------------------------------------------------------
    sorted_businesses = sorted(businesses_raw, key=lambda biz: biz["review_count"], reverse=True)
    
    # -----------------------------------------------------------------------------
    # Limit the results to the top 3.
    # -----------------------------------------------------------------------------
    businesses = sorted_businesses[:3]
    
    # -----------------------------------------------------------------------------
    # Cache and log the results.
    # -----------------------------------------------------------------------------
    if businesses:
        save_businesses(category_name, country, businesses, param_hash)
        log_message(f"✅ Scraped {len(businesses)} businesses for {category_name} in {country} (verified={verified}, claimed={claimed}, min_trustscore={min_trustscore})")
    else:
        log_message(f"ℹ️ No businesses found for {category_name} in {country} matching criteria.")
    
    return businesses

# -----------------------------------------------------------------------------
# Main Program Flow
# -----------------------------------------------------------------------------
def main():
    # STEP 1: Run three sequential scraping runs for categories.
    # (Not used in manual mode; STEP_ONE is False.)
    if STEP_ONE:
        scrape_categories_main()

    # STEP 2: Retrieve a flattened, alphabetized list of valid categories.
    if STEP_TWO:
        if not STEP_ONE:
            # When using manual categories, convert each manual category string into a (name, url) tuple.
            # Build the URL as "https://www.trustpilot.com/categories/<category>".
            cats = [(cat, f"https://www.trustpilot.com/categories/{cat}") for cat in MANUAL_CATEGORIES]
        else:
            cats = get_all_categories_flat()
        if not cats:
            log_message("⚠ No categories found after scraping. Exiting.")
            return

        # Display available categories for user selection.
        print("\nAvailable Categories:")
        for idx, (name, url) in enumerate(cats, start=1):
            print(f"{idx}. {name}")
        choice = input("Enter the number of the category you want to scrape: ").strip()
        try:
            choice = int(choice)
            if choice < 1 or choice > len(cats):
                print("Invalid choice.")
                return
        except ValueError:
            print("Invalid input.")
            return
        selected_category, selected_url = cats[choice - 1]
        log_message(f"Selected Category: {selected_category} ({selected_url})")

    # STEP 3: Prompt the user for filtering parameters.
    if STEP_THREE:
        verified_input = input("Should businesses be verified? (y/n): ").strip().lower()
        verified = True if verified_input == 'y' else False
        claimed_input = input("Should businesses be claimed? (y/n): ").strip().lower()
        claimed = True if claimed_input == 'y' else False
        min_trustscore_input = input("Enter minimum TrustScore (3.0, 4.0, 4.5 or press Enter for ANY): ").strip()
        if min_trustscore_input in {"3.0", "4.0", "4.5"}:
            min_trustscore = float(min_trustscore_input)
        else:
            min_trustscore = None

    # STEP 4: Retrieve (or scrape & cache) the list of countries.
    # Here we pass the selected category URL so that we fetch the page source from that page.
    if STEP_FOUR:
        countries = get_countries(selected_url)
        if not countries:
            log_message("⚠ No countries available. Exiting.")
            return

    # STEP 5: For each country, scrape the top businesses for the selected category sequentially.
    if STEP_FIVE:
        log_message(f"\n🚀 Scraping businesses for category '{selected_category}' across {len(countries)} countries...")
        results = {}
        for country_code, country_name in countries:
            businesses = scrape_top_businesses_for_country(selected_category, selected_url,
                                                           country_code, verified, claimed, min_trustscore)
            results[country_code] = businesses

    # STEP 6: Display a summary of the scraped results.
    if STEP_SIX:
        print("\nScraping Results:")
        for country_code, businesses in results.items():
            country_name = next((name for code, name in countries if code == country_code), country_code)
            print(f"\nCountry: {country_name} ({country_code})")
            if businesses:
                for biz in businesses:
                    print(f" - {biz['name']} (Rating: {biz['rating']}, Claimed: {biz['claimed']}, Verified: {biz['verified']})")
            else:
                print(" No matching businesses found.")
        log_message("\n🚀 Business Scraping complete.")

if __name__ == '__main__':
    main()
